
# Major Project: Seasonal Agriculture Performance Analysis

**Platform:** Google Colab / Jupyter Notebook  
**Dataset:** `seasonal_agriculture_performance_dataset(1).csv`

This project analyzes agricultural performance across seasons, crops and regions, including environmental conditions, resource usage, yield, production and economic outcomes.



## 1. Problem Statement

Agricultural activities are influenced by seasonal variations in environmental conditions, farming practices, resource availability and market conditions. This project investigates seasonal differences in agricultural performance by identifying meaningful patterns, trends, relationships and variations in the available data.



## 2. Objectives

- Explore and understand the dataset.
- Clean and prepare the data.
- Compare agricultural performance across seasons.
- Analyze crop and regional differences.
- Examine environmental conditions and resource usage.
- Study yield, production, revenue, cost and profit.
- Investigate relationships between agricultural factors and outcomes.
- Identify unusual patterns and outliers.
- Apply statistical and visualization techniques.
- Develop evidence-based conclusions and recommendations.


In [ ]:

# Cell 1 — Import libraries

!pip -q install scipy seaborn openpyxl

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully.")


In [ ]:

# Cell 2 — Upload dataset in Google Colab

from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))


In [ ]:

# Cell 3 — Load dataset

file_name = "seasonal_agriculture_performance_dataset(1).csv"

if not os.path.exists(file_name):
    csv_files = [f for f in os.listdir(".") if f.lower().endswith(".csv")]
    if not csv_files:
        raise FileNotFoundError("No CSV file found. Upload the dataset first.")
    file_name = csv_files[0]

df = pd.read_csv(file_name)

print("File:", file_name)
print("Rows:", f"{df.shape[0]:,}")
print("Columns:", df.shape[1])

display(df.head())


In [ ]:

# Cell 4 — Dataset structure

print("Shape:", df.shape)

print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(i, col)

print("\nData types:")
display(df.dtypes.to_frame("Data_Type"))


In [ ]:

# Cell 5 — Statistical overview

display(df.describe(include="all").T)


In [ ]:

# Cell 6 — Missing values

missing = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": df.isna().mean() * 100
}).sort_values("Missing_Count", ascending=False)

display(missing[missing["Missing_Count"] > 0])


In [ ]:

# Cell 7 — Duplicate records

print("Duplicate rows:", df.duplicated().sum())


In [ ]:

# Cell 8 — Categorical variables

categorical_cols = ["State", "District", "Crop", "Season", "Irrigation_Method"]

for col in categorical_cols:
    print(f"\n{col} — {df[col].nunique()} unique values")
    print(df[col].value_counts(dropna=False).head(15))


## 3. Data Cleaning and Preparation

In [ ]:

# Cell 9 — Clean the dataset

data = df.copy()

# Remove exact duplicates
data = data.drop_duplicates().reset_index(drop=True)

# Clean text fields
for col in categorical_cols:
    data[col] = data[col].astype("string").str.strip()

# Convert numeric fields
for col in data.select_dtypes(include=np.number).columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Numeric missing values → median
for col in data.select_dtypes(include=np.number).columns:
    if data[col].isna().any():
        data[col] = data[col].fillna(data[col].median())

# Categorical missing values → mode
for col in data.select_dtypes(include=["object", "string", "category"]).columns:
    if data[col].isna().any():
        mode = data[col].mode(dropna=True)
        if len(mode):
            data[col] = data[col].fillna(mode.iloc[0])

print("Rows after cleaning:", len(data))
print("Missing values remaining:", data.isna().sum().sum())
print("Duplicates remaining:", data.duplicated().sum())


In [ ]:

# Cell 10 — Validate important ranges

range_checks = {
    "Farm_Area_Hectares": (0, np.inf),
    "Rainfall_mm": (0, np.inf),
    "Humidity_pct": (0, 100),
    "Sunlight_Hours_Day": (0, 24),
    "Soil_Moisture_pct": (0, 100),
    "Seed_Quality_Score": (0, 1),
    "Yield_Tonnes_Ha": (0, np.inf),
    "Production_Tonnes": (0, np.inf),
    "Total_Cost_INR": (0, np.inf),
    "Revenue_INR": (0, np.inf),
    "Water_Used_m3": (0, np.inf),
    "Water_Efficiency_t_per_1000m3": (0, np.inf),
    "Disease_Pest_Risk_pct": (0, 100)
}

for col, (low, high) in range_checks.items():
    if col in data.columns:
        bad = ((data[col] < low) | (data[col] > high)).sum()
        print(f"{col}: {bad} outside expected range")


## 4. Feature Engineering

In [ ]:

# Cell 11 — Create additional analytical variables

data["Profit_Margin_pct"] = np.where(
    data["Revenue_INR"] > 0,
    data["Profit_INR"] / data["Revenue_INR"] * 100,
    np.nan
)

data["Cost_per_Tonne_INR"] = np.where(
    data["Production_Tonnes"] > 0,
    data["Total_Cost_INR"] / data["Production_Tonnes"],
    np.nan
)

data["Revenue_per_Hectare_INR"] = np.where(
    data["Farm_Area_Hectares"] > 0,
    data["Revenue_INR"] / data["Farm_Area_Hectares"],
    np.nan
)

data["Profit_per_Hectare_INR"] = np.where(
    data["Farm_Area_Hectares"] > 0,
    data["Profit_INR"] / data["Farm_Area_Hectares"],
    np.nan
)

data["Production_per_Hectare_Tonnes"] = np.where(
    data["Farm_Area_Hectares"] > 0,
    data["Production_Tonnes"] / data["Farm_Area_Hectares"],
    np.nan
)

display(data.head())


## 5. Exploratory Data Analysis

In [ ]:

# Cell 12 — Season distribution

season_counts = data["Season"].value_counts()

plt.figure(figsize=(8, 5))
sns.barplot(x=season_counts.index, y=season_counts.values)
plt.title("Number of Farm Records by Season")
plt.xlabel("Season")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()

display(season_counts.to_frame("Record_Count"))


In [ ]:

# Cell 13 — Crop distribution

crop_counts = data["Crop"].value_counts()

plt.figure(figsize=(10, 5))
sns.barplot(x=crop_counts.index, y=crop_counts.values)
plt.title("Number of Farm Records by Crop")
plt.xlabel("Crop")
plt.ylabel("Number of Records")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:

# Cell 14 — State distribution

state_counts = data["State"].value_counts().head(15)

plt.figure(figsize=(10, 6))
sns.barplot(y=state_counts.index, x=state_counts.values)
plt.title("Top States by Number of Farm Records")
plt.xlabel("Number of Records")
plt.ylabel("State")
plt.tight_layout()
plt.show()


## 6. Key Question 1 — How does agricultural performance vary across seasons?

In [ ]:

# Cell 15 — Seasonal performance summary

season_summary = data.groupby("Season").agg(
    Farms=("Farm_ID", "count"),
    Avg_Yield_Tonnes_Ha=("Yield_Tonnes_Ha", "mean"),
    Avg_Production_Tonnes=("Production_Tonnes", "mean"),
    Avg_Revenue_INR=("Revenue_INR", "mean"),
    Avg_Cost_INR=("Total_Cost_INR", "mean"),
    Avg_Profit_INR=("Profit_INR", "mean"),
    Avg_Water_Used_m3=("Water_Used_m3", "mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Avg_Disease_Pest_Risk_pct=("Disease_Pest_Risk_pct", "mean")
).sort_values("Avg_Profit_INR", ascending=False)

display(season_summary.round(2))


In [ ]:

# Cell 16 — Average yield by season

plt.figure(figsize=(8, 5))
sns.barplot(data=data, x="Season", y="Yield_Tonnes_Ha", estimator=np.mean, errorbar=None)
plt.title("Average Crop Yield by Season")
plt.xlabel("Season")
plt.ylabel("Average Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()

best_yield_season = season_summary["Avg_Yield_Tonnes_Ha"].idxmax()
print("Highest average yield season:", best_yield_season)


In [ ]:

# Cell 17 — Seasonal KPI boxplots

metrics = [
    "Yield_Tonnes_Ha",
    "Production_Tonnes",
    "Revenue_INR",
    "Total_Cost_INR",
    "Profit_INR",
    "Water_Used_m3",
    "Water_Efficiency_t_per_1000m3",
    "Disease_Pest_Risk_pct"
]

for metric in metrics:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=data, x="Season", y=metric)
    plt.title(f"{metric} by Season")
    plt.xlabel("Season")
    plt.ylabel(metric)
    plt.tight_layout()
    plt.show()


## 7. Key Question 2 — Which crops perform best in each season?

In [ ]:

# Cell 18 — Crop-season performance

crop_season = data.groupby(["Season", "Crop"]).agg(
    Farms=("Farm_ID", "count"),
    Avg_Yield=("Yield_Tonnes_Ha", "mean"),
    Avg_Profit=("Profit_INR", "mean"),
    Avg_Revenue=("Revenue_INR", "mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean")
).reset_index()

display(crop_season.sort_values(["Season", "Avg_Yield"], ascending=[True, False]).round(2))


In [ ]:

# Cell 19 — Best crop by yield in each season

best_crop_by_season = (
    crop_season.sort_values(["Season", "Avg_Yield"], ascending=[True, False])
    .groupby("Season")
    .head(1)
)

display(best_crop_by_season.round(2))


In [ ]:

# Cell 20 — Crop × Season yield heatmap

yield_pivot = data.pivot_table(
    index="Crop",
    columns="Season",
    values="Yield_Tonnes_Ha",
    aggfunc="mean"
)

plt.figure(figsize=(10, 7))
sns.heatmap(yield_pivot, annot=True, fmt=".2f")
plt.title("Average Yield by Crop and Season")
plt.xlabel("Season")
plt.ylabel("Crop")
plt.tight_layout()
plt.show()


## 8. Key Question 3 — Which regions have stronger agricultural performance?

In [ ]:

# Cell 21 — State-wise performance

state_summary = data.groupby("State").agg(
    Farms=("Farm_ID", "count"),
    Avg_Yield=("Yield_Tonnes_Ha", "mean"),
    Avg_Production=("Production_Tonnes", "mean"),
    Avg_Revenue=("Revenue_INR", "mean"),
    Avg_Profit=("Profit_INR", "mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean")
).sort_values("Avg_Profit", ascending=False)

display(state_summary.round(2))


In [ ]:

# Cell 22 — Top states by average yield

top_states = state_summary.sort_values("Avg_Yield", ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(y=top_states.index, x=top_states["Avg_Yield"])
plt.title("Top 10 States by Average Yield")
plt.xlabel("Average Yield (Tonnes/Ha)")
plt.ylabel("State")
plt.tight_layout()
plt.show()


## 9. Key Question 4 — What is the relationship between environmental conditions and yield?

In [ ]:

# Cell 23 — Environmental correlations

environmental_cols = [
    "Rainfall_mm",
    "Avg_Temperature_C",
    "Humidity_pct",
    "Sunlight_Hours_Day",
    "Soil_pH",
    "Soil_Moisture_pct"
]

environment_corr = (
    data[environmental_cols + ["Yield_Tonnes_Ha"]]
    .corr()["Yield_Tonnes_Ha"]
    .drop("Yield_Tonnes_Ha")
    .sort_values(key=abs, ascending=False)
)

display(environment_corr.to_frame("Correlation_with_Yield"))


In [ ]:

# Cell 24 — Environmental factors vs yield

for col in environmental_cols:
    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=data, x=col, y="Yield_Tonnes_Ha", alpha=0.5)
    sns.regplot(data=data, x=col, y="Yield_Tonnes_Ha", scatter=False)
    plt.title(f"{col} vs Yield")
    plt.xlabel(col)
    plt.ylabel("Yield (Tonnes/Ha)")
    plt.tight_layout()
    plt.show()


## 10. Key Question 5 — How do resources and irrigation affect performance?

In [ ]:

# Cell 25 — Resource usage by season

resource_cols = [
    "Nitrogen_kg_ha",
    "Phosphorus_kg_ha",
    "Potassium_kg_ha",
    "Fertilizer_kg_ha",
    "Pesticide_Litre_ha",
    "Water_Used_m3"
]

resource_season = data.groupby("Season")[resource_cols].mean()
display(resource_season.round(2))


In [ ]:

# Cell 26 — Irrigation method comparison

irrigation_summary = data.groupby("Irrigation_Method").agg(
    Farms=("Farm_ID", "count"),
    Avg_Yield=("Yield_Tonnes_Ha", "mean"),
    Avg_Profit=("Profit_INR", "mean"),
    Avg_Water_Used=("Water_Used_m3", "mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean")
).sort_values("Avg_Yield", ascending=False)

display(irrigation_summary.round(2))

plt.figure(figsize=(9, 5))
sns.barplot(
    data=irrigation_summary.reset_index(),
    x="Irrigation_Method",
    y="Avg_Yield"
)
plt.title("Average Yield by Irrigation Method")
plt.xlabel("Irrigation Method")
plt.ylabel("Average Yield (Tonnes/Ha)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:

# Cell 27 — Resources vs yield/profit correlation

resource_outcomes = [
    "Fertilizer_kg_ha",
    "Pesticide_Litre_ha",
    "Water_Used_m3",
    "Nitrogen_kg_ha",
    "Phosphorus_kg_ha",
    "Potassium_kg_ha",
    "Seed_Quality_Score",
    "Yield_Tonnes_Ha",
    "Profit_INR"
]

resource_corr = data[resource_outcomes].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(resource_corr, annot=True, fmt=".2f", center=0)
plt.title("Resource and Outcome Correlation")
plt.tight_layout()
plt.show()


## 11. Key Question 6 — How does seed quality relate to yield?

In [ ]:

# Cell 28 — Seed quality analysis

r = data["Seed_Quality_Score"].corr(data["Yield_Tonnes_Ha"])

plt.figure(figsize=(7, 5))
sns.scatterplot(data=data, x="Seed_Quality_Score", y="Yield_Tonnes_Ha", alpha=0.5)
sns.regplot(data=data, x="Seed_Quality_Score", y="Yield_Tonnes_Ha", scatter=False)
plt.title("Seed Quality Score vs Yield")
plt.xlabel("Seed Quality Score")
plt.ylabel("Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()

print(f"Correlation between seed quality and yield: {r:.3f}")


## 12. Key Question 7 — How do economic outcomes vary across seasons?

In [ ]:

# Cell 29 — Economic performance

economic = data.groupby("Season").agg(
    Avg_Revenue=("Revenue_INR", "mean"),
    Avg_Cost=("Total_Cost_INR", "mean"),
    Avg_Profit=("Profit_INR", "mean"),
    Avg_Profit_Margin=("Profit_Margin_pct", "mean"),
    Avg_Revenue_per_Hectare=("Revenue_per_Hectare_INR", "mean"),
    Avg_Profit_per_Hectare=("Profit_per_Hectare_INR", "mean")
).sort_values("Avg_Profit", ascending=False)

display(economic.round(2))


In [ ]:

# Cell 30 — Revenue, cost and profit

economic[["Avg_Revenue", "Avg_Cost", "Avg_Profit"]].plot(
    kind="bar", figsize=(10, 6)
)

plt.title("Average Revenue, Cost and Profit by Season")
plt.xlabel("Season")
plt.ylabel("Amount (INR)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 13. Key Question 8 — Which crops are most profitable?

In [ ]:

# Cell 31 — Crop profitability

crop_profit = data.groupby("Crop").agg(
    Farms=("Farm_ID", "count"),
    Avg_Revenue=("Revenue_INR", "mean"),
    Avg_Cost=("Total_Cost_INR", "mean"),
    Avg_Profit=("Profit_INR", "mean"),
    Avg_Profit_Margin=("Profit_Margin_pct", "mean"),
    Avg_Yield=("Yield_Tonnes_Ha", "mean")
).sort_values("Avg_Profit", ascending=False)

display(crop_profit.round(2))

top = crop_profit.head(10)

plt.figure(figsize=(10, 6))
sns.barplot(y=top.index, x=top["Avg_Profit"])
plt.title("Top Crops by Average Profit")
plt.xlabel("Average Profit (INR)")
plt.ylabel("Crop")
plt.tight_layout()
plt.show()


## 14. Key Question 9 — Which seasons use water most efficiently?

In [ ]:

# Cell 32 — Water efficiency

water_season = data.groupby("Season").agg(
    Avg_Water_Used=("Water_Used_m3", "mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Avg_Yield=("Yield_Tonnes_Ha", "mean")
).sort_values("Avg_Water_Efficiency", ascending=False)

display(water_season.round(3))

plt.figure(figsize=(8, 5))
sns.barplot(
    data=water_season.reset_index(),
    x="Season",
    y="Avg_Water_Efficiency"
)
plt.title("Average Water Efficiency by Season")
plt.xlabel("Season")
plt.ylabel("Tonnes per 1,000 m³")
plt.tight_layout()
plt.show()


## 15. Key Question 10 — How does disease/pest risk vary?

In [ ]:

# Cell 33 — Disease/pest risk

risk_season = (
    data.groupby("Season")["Disease_Pest_Risk_pct"]
    .mean()
    .sort_values(ascending=False)
)

display(risk_season.to_frame("Average_Risk_pct").round(2))

plt.figure(figsize=(8, 5))
sns.barplot(x=risk_season.index, y=risk_season.values)
plt.title("Average Disease/Pest Risk by Season")
plt.xlabel("Season")
plt.ylabel("Average Risk (%)")
plt.tight_layout()
plt.show()

risk_crop = (
    data.groupby("Crop")["Disease_Pest_Risk_pct"]
    .mean()
    .sort_values(ascending=False)
)

display(risk_crop.to_frame("Average_Risk_pct").round(2))


## 16. Overall Correlation Analysis

In [ ]:

# Cell 34 — Full numeric correlation heatmap

numeric_cols = data.select_dtypes(include=np.number).columns
corr = data[numeric_cols].corr()

plt.figure(figsize=(18, 14))
sns.heatmap(corr, center=0)
plt.title("Correlation Heatmap of Numeric Variables")
plt.tight_layout()
plt.show()


In [ ]:

# Cell 35 — Strongest variables associated with yield and profit

yield_corr = (
    data[numeric_cols]
    .corr()["Yield_Tonnes_Ha"]
    .drop("Yield_Tonnes_Ha")
    .sort_values(key=abs, ascending=False)
)

profit_corr = (
    data[numeric_cols]
    .corr()["Profit_INR"]
    .drop("Profit_INR")
    .sort_values(key=abs, ascending=False)
)

print("Top variables associated with Yield:")
display(yield_corr.head(10).to_frame("Correlation"))

print("Top variables associated with Profit:")
display(profit_corr.head(10).to_frame("Correlation"))


## 17. Outlier Analysis

In [ ]:

# Cell 36 — IQR outlier detection

outlier_cols = [
    "Yield_Tonnes_Ha",
    "Production_Tonnes",
    "Revenue_INR",
    "Total_Cost_INR",
    "Profit_INR",
    "Water_Used_m3",
    "Disease_Pest_Risk_pct"
]

results = []

for col in outlier_cols:
    q1 = data[col].quantile(0.25)
    q3 = data[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((data[col] < lower) | (data[col] > upper)).sum()
    results.append([col, q1, q3, lower, upper, count])

outlier_table = pd.DataFrame(
    results,
    columns=["Variable", "Q1", "Q3", "Lower_Bound", "Upper_Bound", "Outlier_Count"]
)

display(outlier_table.round(2))


In [ ]:

# Cell 37 — Visualize important outliers

for col in ["Yield_Tonnes_Ha", "Profit_INR", "Water_Used_m3"]:
    plt.figure(figsize=(9, 4))
    sns.boxplot(x=data[col])
    plt.title(f"Outlier Detection — {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()


## 18. Statistical Analysis — Do seasons differ significantly in yield?

In [ ]:

# Cell 38 — ANOVA and Kruskal-Wallis tests

groups = [
    group["Yield_Tonnes_Ha"].dropna().values
    for _, group in data.groupby("Season")
]

anova_stat, anova_p = stats.f_oneway(*groups)
kw_stat, kw_p = stats.kruskal(*groups)

print(f"ANOVA statistic: {anova_stat:.4f}")
print(f"ANOVA p-value:   {anova_p:.6f}")

print(f"\nKruskal-Wallis statistic: {kw_stat:.4f}")
print(f"Kruskal-Wallis p-value:   {kw_p:.6f}")

alpha = 0.05

if anova_p < alpha:
    print("\nANOVA: Significant evidence of a difference in mean yield among seasons.")
else:
    print("\nANOVA: Insufficient evidence of a difference in mean yield among seasons.")

if kw_p < alpha:
    print("Kruskal-Wallis: Yield distributions differ significantly among seasons.")
else:
    print("Kruskal-Wallis: Insufficient evidence of different yield distributions among seasons.")


## 19. Pairwise Seasonal Comparison

In [ ]:

# Cell 39 — Pairwise Mann-Whitney U tests with Bonferroni correction

seasons = data["Season"].dropna().unique().tolist()
pairwise_results = []

for i in range(len(seasons)):
    for j in range(i + 1, len(seasons)):
        s1, s2 = seasons[i], seasons[j]

        x = data.loc[data["Season"] == s1, "Yield_Tonnes_Ha"].dropna()
        y = data.loc[data["Season"] == s2, "Yield_Tonnes_Ha"].dropna()

        u, p = stats.mannwhitneyu(x, y, alternative="two-sided")
        pairwise_results.append([s1, s2, u, p])

pairwise = pd.DataFrame(
    pairwise_results,
    columns=["Season_1", "Season_2", "U_Statistic", "Raw_p_value"]
)

m = len(pairwise)
pairwise["Bonferroni_p_value"] = np.minimum(pairwise["Raw_p_value"] * m, 1.0)
pairwise["Significant_at_0.05"] = pairwise["Bonferroni_p_value"] < 0.05

display(pairwise.sort_values("Bonferroni_p_value").round(6))


## 20. Automatically Generated Key Findings

In [ ]:

# Cell 40 — Generate findings from actual calculated results

best_yield = season_summary["Avg_Yield_Tonnes_Ha"].idxmax()
best_profit = season_summary["Avg_Profit_INR"].idxmax()
best_water = season_summary["Avg_Water_Efficiency"].idxmax()
lowest_risk = season_summary["Avg_Disease_Pest_Risk_pct"].idxmin()

best_crop = crop_profit["Avg_Profit"].idxmax()
best_state = state_summary["Avg_Profit"].idxmax()

strongest_yield_factor = yield_corr.index[0]
strongest_yield_corr = yield_corr.iloc[0]

print("KEY FINDINGS")
print("-" * 70)
print(f"1. Highest average yield season: {best_yield}")
print(f"   Average yield: {season_summary.loc[best_yield, 'Avg_Yield_Tonnes_Ha']:.2f} tonnes/ha")

print(f"\n2. Highest average profit season: {best_profit}")
print(f"   Average profit: ₹{season_summary.loc[best_profit, 'Avg_Profit_INR']:,.2f}")

print(f"\n3. Highest water-efficiency season: {best_water}")
print(f"   Efficiency: {season_summary.loc[best_water, 'Avg_Water_Efficiency']:.3f} tonnes/1,000 m³")

print(f"\n4. Lowest disease/pest risk season: {lowest_risk}")
print(f"   Risk: {season_summary.loc[lowest_risk, 'Avg_Disease_Pest_Risk_pct']:.2f}%")

print(f"\n5. Most profitable crop on average: {best_crop}")
print(f"   Average profit: ₹{crop_profit.loc[best_crop, 'Avg_Profit']:,.2f}")

print(f"\n6. Highest average-profit state: {best_state}")
print(f"   Average profit: ₹{state_summary.loc[best_state, 'Avg_Profit']:,.2f}")

print(f"\n7. Strongest absolute numeric correlation with yield: {strongest_yield_factor}")
print(f"   Correlation: {strongest_yield_corr:.3f}")


## 21. Evidence-Based Recommendations

In [ ]:

# Cell 41 — Generate recommendations from the results

recommendations = [
    f"Prioritize seasonal planning around {best_yield}, which has the highest observed average yield.",
    f"Study the practices associated with {best_profit}, which has the highest observed average profit.",
    f"Evaluate water-management practices associated with {best_water}, the highest water-efficiency season.",
    f"Strengthen disease/pest monitoring in seasons with comparatively higher observed risk.",
    f"Study {best_crop}, the crop with the highest average profit, before considering wider adoption.",
    f"Investigate {strongest_yield_factor} further because it has the strongest absolute numeric correlation with yield."
]

for i, item in enumerate(recommendations, 1):
    print(f"{i}. {item}")



## 22. Interpretation Note

Correlation indicates association, not causation. A strong correlation does not prove that changing one factor alone will change yield or profit.

Seasonal differences may also be affected by crop mix, geography, farming practices, soil conditions, market conditions and other variables. Therefore, the recommendations are evidence-based areas for further investigation rather than guaranteed causal effects.


## 23. Final Conclusion

In [ ]:

# Cell 42 — Generate final conclusion

overall_yield = data["Yield_Tonnes_Ha"].mean()
overall_profit = data["Profit_INR"].mean()
overall_water_eff = data["Water_Efficiency_t_per_1000m3"].mean()

conclusion = (
    "The analysis covered "
    + f"{len(data):,} agricultural records across {data['Season'].nunique()} seasons, "
    + f"{data['Crop'].nunique()} crops and {data['State'].nunique()} states.\n\n"
    + f"The overall average yield was {overall_yield:.2f} tonnes/ha and the overall average "
    + f"profit was ₹{overall_profit:,.2f}. "
    + f"{best_yield} recorded the highest average yield, while {best_profit} recorded "
    + "the highest average profit.\n\n"
    + f"{best_water} had the highest average water efficiency, while {lowest_risk} "
    + "had the lowest average disease/pest risk. "
    + f"The strongest absolute numeric correlation with yield was {strongest_yield_factor} "
    + f"(correlation = {strongest_yield_corr:.3f}).\n\n"
    + "Overall, the project demonstrates how seasonal agricultural data can be used to compare "
    + "performance, understand resource and environmental patterns, examine economic outcomes "
    + "and support evidence-based seasonal agricultural planning."
)

display(Markdown("### Conclusion\n\n" + conclusion.replace("\n\n", "\n\n")))


## 24. Export Results

In [ ]:

# Cell 43 — Save important project outputs

data.to_csv("cleaned_seasonal_agriculture_dataset.csv", index=False)
season_summary.to_csv("seasonal_performance_summary.csv")
crop_profit.to_csv("crop_profitability_summary.csv")
state_summary.to_csv("state_performance_summary.csv")
pairwise.to_csv("pairwise_seasonal_yield_tests.csv", index=False)

print("Exported successfully:")
print("• cleaned_seasonal_agriculture_dataset.csv")
print("• seasonal_performance_summary.csv")
print("• crop_profitability_summary.csv")
print("• state_performance_summary.csv")
print("• pairwise_seasonal_yield_tests.csv")


In [ ]:

# Cell 44 — Optional: download the cleaned dataset in Colab

from google.colab import files

files.download("cleaned_seasonal_agriculture_dataset.csv")



# End of Project

### Suggested PPT structure

1. Title
2. Introduction
3. Problem Statement
4. Objectives
5. Dataset Description
6. Data Cleaning
7. Analytical Questions
8. Seasonal Performance
9. Crop Analysis
10. Regional Analysis
11. Environmental Analysis
12. Resource and Irrigation Analysis
13. Economic Analysis
14. Water Efficiency
15. Disease/Pest Risk
16. Correlation Analysis
17. Statistical Testing
18. Key Findings
19. Recommendations
20. Conclusion

Use screenshots of the important tables and charts generated by this notebook in your final PPT.
